# Notebook 7 — Improving the Best Model

## Story
Across all previous notebooks, **EfficientNetV2-S** (or whichever backbone
achieved the highest micro-F1 in Notebook 5) consistently emerged as the
strongest performer.  Now we ask: *how far can targeted engineering push it?*

We test five independent improvements and combine the most effective ones:

| # | Technique | Rationale |
|---|---|---|
| 1 | **Asymmetric Loss (ASL)** | In multi-label problems most labels are absent. BCE treats hard positives and easy negatives equally. ASL down-weights easy negatives with a class-specific focal term. |
| 2 | **RandAugment + RandomErasing** | Stronger, principled augmentation policy; forces model to use global context. |
| 3 | **Stratified Split** | Guarantees rare label combos are in both val and test splits. |
| 4 | **Weighted Random Sampling** | Oversamples under-represented label combos during training to reduce class imbalance. |
| 5 | **Per-class Threshold Tuning** | Instead of a global 0.5 threshold, find the optimal threshold per class on the validation set. |

**Labels (12):** `pen, paper, book, clock, phone, laptop, chair, desk, bottle, keychain, backpack, calculator`

## 1. Imports & Setup

In [ ]:
import sys
sys.path.insert(0, "../")
sys.path.insert(0, "../experiments")

import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from collections import defaultdict
from torchvision import models as tv_models
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, subsample_subset,
    print_model_info, save_checkpoint, load_checkpoint,
    plot_training_history,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices,
    compute_multilabel_metrics, NUM_LABELS, METRIC_KEYS,
    NORM_MEAN, NORM_STD, TransformSubset,
)

SEED   = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 2. Config & Best Backbone

In [ ]:
BASE_DATA_DIR   = "../data/aggregated"
IMAGE_SIZE      = 224
BATCH_SIZE      = 32
SPLIT           = [0.7, 0.15, 0.15]
USE_SUBSET      = False
CHECKPOINT_DIR  = Path("../checkpoints")

# ── The best backbone from Notebook 5 ─────────────────────────────────────────
# Change BEST_ARCH if a different model had a higher F1 in your runs.
BEST_ARCH = "efficientnet_v2_s"    # or "convnext_tiny", "resnet50"

print(f"Using backbone: {BEST_ARCH}")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")

## 3. Helper: Build Best Backbone

In [ ]:
class TransferModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        return self.head(self.backbone(x))

    def set_backbone_trainable(self, flag):
        for p in self.backbone.parameters():
            p.requires_grad = flag


def build_best_model(num_labels: int) -> TransferModel:
    if BEST_ARCH == "efficientnet_v2_s":
        m        = tv_models.efficientnet_v2_s(weights=tv_models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, num_labels))
    elif BEST_ARCH == "convnext_tiny":
        m        = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Sequential(nn.LayerNorm(768), nn.Dropout(0.2), nn.Linear(768, num_labels))
    elif BEST_ARCH == "resnet50":
        m        = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
        backbone = nn.Sequential(*list(m.children())[:-1], nn.Flatten())
        head     = nn.Sequential(nn.Dropout(0.3), nn.Linear(2048, num_labels))
    else:
        raise ValueError(f"Unknown arch: {BEST_ARCH}")

    model = TransferModel(backbone, head)
    for layer in model.head.modules():
        if isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, 0, 0.01)
            nn.init.zeros_(layer.bias)
    return model


print_model_info(build_best_model(NUM_LABELS).to(DEVICE))

## 4. Improvement 1 — Asymmetric Loss (ASL)

Standard BCE assigns equal weight to all negatives, but in multi-label
classification, most labels are absent in any given image — the network
is flooded with easy negatives whose gradients dominate and drown out
the hard, informative positives.

ASL decouples the focal parameter for positives (`gamma_pos`) and negatives
(`gamma_neg`) and optionally shifts negative probabilities by `clip` to
further suppress trivially-negative predictions.

In [ ]:
class AsymmetricLoss(nn.Module):
    """Asymmetric Loss (Ben-Baruch et al., 2020)."""

    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps

    def forward(self, logits, targets):
        probs_pos = torch.sigmoid(logits)
        probs_neg = 1.0 - probs_pos
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)
        loss_pos = targets       * torch.log(probs_pos.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(probs_neg.clamp(min=self.eps))
        loss     = loss_pos + loss_neg
        pt    = probs_pos * targets + probs_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        loss  = loss * (1 - pt).pow(gamma)
        return -loss.mean()

## 5. Improvement 2 — RandAugment + RandomErasing

In [ ]:
randaug_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    RandomErasing(p=0.25, scale=(0.02, 0.1)),
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

## 6. Improvement 3 — Stratified Split

In [ ]:
def stratified_split_multilabel(dataset, split, seed=42):
    combo_to_indices = defaultdict(list)
    for i in range(len(dataset)):
        _, target = dataset[i]
        combo_to_indices[tuple(target.int().tolist())].append(i)

    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for indices in combo_to_indices.values():
        indices = np.array(indices)
        rng.shuffle(indices)
        n       = len(indices)
        n_val   = max(1 if n >= 3 else 0, round(split[1] * n))
        n_test  = max(1 if n >= 3 else 0, round(split[2] * n))
        n_train = max(0, n - n_val - n_test)
        train_idx.extend(indices[:n_train].tolist())
        val_idx.extend(indices[n_train: n_train + n_val].tolist())
        test_idx.extend(indices[n_train + n_val:].tolist())
    return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)


full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = stratified_split_multilabel(full_dataset, SPLIT, SEED)
print(f"Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 7. Improvement 4 — Weighted Random Sampling

Some label combinations are very rare in the training set.  Standard random
batches rarely contain these combos, so the model never learns them well.

`WeightedRandomSampler` assigns each sample a weight inversely proportional
to the frequency of its label combination, so rare combos appear in batches
as often as common ones.

In [ ]:
def make_weighted_sampler(subset):
    """Build a WeightedRandomSampler that oversamples rare label combos."""
    combo_count = defaultdict(int)
    combos      = []
    for i in range(len(subset)):
        _, target = subset[i]
        combo     = tuple(target.int().tolist())
        combos.append(combo)
        combo_count[combo] += 1

    weights = [1.0 / combo_count[c] for c in combos]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


sampler = make_weighted_sampler(train_raw)

train_ds = TransformSubset(train_raw, transform=randaug_train_transform)
val_ds   = TransformSubset(val_raw,   transform=eval_transform)
test_ds  = TransformSubset(test_raw,  transform=eval_transform)

# Use sampler instead of shuffle=True
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

## 8. Train with All Improvements Combined

In [ ]:
FREEZE_EPOCHS   = 3
UNFREEZE_EPOCHS = 20
LR_HEAD         = 1e-3
LR_BACKBONE     = 1e-4
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
THRESHOLD       = 0.5
criterion       = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
IMPROVED_PATH   = CHECKPOINT_DIR / f"improved_{BEST_ARCH}.pth"


def run_epoch(model, loader, optimizer=None, train=True):
    model.train(train)
    all_logits, all_preds, all_labels = [], [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
                )
                optimizer.step()
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()
        all_logits.append(logits.detach().cpu())
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())
    return compute_multilabel_metrics(
        torch.cat(all_labels), torch.cat(all_preds), logits=torch.cat(all_logits)
    )


model    = build_best_model(NUM_LABELS).to(DEVICE)
model.set_backbone_trainable(False)
history  = {k: {"train": [], "val": []} for k in METRIC_KEYS}
best_f1, best_state = -1.0, None
best_val_probs_list = None

for epoch in range(1, FREEZE_EPOCHS + UNFREEZE_EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:
        model.set_backbone_trainable(True)
        print(f"  [Epoch {epoch}] Backbone unfrozen.")

    optimizer = optim.Adam([
        {"params": model.head.parameters(),     "lr": LR_HEAD},
        {"params": model.backbone.parameters(), "lr": (LR_BACKBONE if epoch > FREEZE_EPOCHS else LR_HEAD)},
    ], weight_decay=WEIGHT_DECAY)

    tr  = run_epoch(model, train_loader, optimizer, train=True)
    val = run_epoch(model, val_loader,   optimizer=None, train=False)

    for k in METRIC_KEYS:
        history[k]["train"].append(tr[k])
        history[k]["val"].append(val[k])

    if val["f1_micro"] > best_f1:
        best_f1    = val["f1_micro"]
        best_state = copy.deepcopy(model.state_dict())

    print(f"  Epoch {epoch:>2}  train_f1={tr['f1_micro']:.4f}  val_f1={val['f1_micro']:.4f}")

save_checkpoint(best_state, IMPROVED_PATH)
print(f"\nImproved model best val F1: {best_f1:.4f}")
plot_training_history(history, FREEZE_EPOCHS + UNFREEZE_EPOCHS,
                      f"Improved {BEST_ARCH}", LR_HEAD, WEIGHT_DECAY)

## 9. Improvement 5 — Per-class Threshold Tuning

A single 0.5 threshold is suboptimal: some classes need higher thresholds
(to reduce FP) while others need lower ones (to increase recall).  We sweep
thresholds on the **validation set** for each class independently.

In [ ]:
model = build_best_model(NUM_LABELS).to(DEVICE)
model.load_state_dict(best_state)
model.eval()

# Collect val probabilities
val_probs_list, val_labels_list = [], []
with torch.no_grad():
    for images, labels in val_loader:
        probs = torch.sigmoid(model(images.to(DEVICE)))
        val_probs_list.append(probs.cpu())
        val_labels_list.append(labels)
val_probs  = torch.cat(val_probs_list)   # (N, C)
val_labels = torch.cat(val_labels_list)  # (N, C)

# Sweep thresholds per class
thresholds = torch.linspace(0.1, 0.9, steps=33)
best_thresholds = torch.ones(NUM_LABELS) * 0.5
for c in range(NUM_LABELS):
    best_f1_c = -1.0
    for t in thresholds:
        preds_c = (val_probs[:, c] >= t).float()
        tp = ((preds_c == 1) & (val_labels[:, c] == 1)).sum().float()
        fp = ((preds_c == 1) & (val_labels[:, c] == 0)).sum().float()
        fn = ((preds_c == 0) & (val_labels[:, c] == 1)).sum().float()
        f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
        if f1 > best_f1_c:
            best_f1_c           = f1
            best_thresholds[c]  = t

print("Per-class optimal thresholds:")
for lbl, t in zip(LABEL_ORDER, best_thresholds):
    print(f"  {lbl:<14}  {t:.2f}")

In [ ]:
# Evaluate on test with per-class thresholds
test_probs_list, test_labels_list = [], []
test_logits_list = []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(DEVICE))
        probs  = torch.sigmoid(logits)
        test_probs_list.append(probs.cpu())
        test_logits_list.append(logits.cpu())
        test_labels_list.append(labels)

test_probs  = torch.cat(test_probs_list)
test_logits = torch.cat(test_logits_list)
test_labels = torch.cat(test_labels_list)

# Global threshold
preds_global = (test_probs >= 0.5).float()
met_global   = compute_multilabel_metrics(test_labels, preds_global, logits=test_logits)

# Per-class threshold
preds_perclass = (test_probs >= best_thresholds.unsqueeze(0)).float()
met_perclass   = compute_multilabel_metrics(test_labels, preds_perclass, logits=test_logits)

print(f"{'Metric':<16} {'Global 0.5':>12}  {'Per-class':>12}")
print("-" * 44)
for k in METRIC_KEYS:
    print(f"{k:<16} {met_global[k]:>12.4f}  {met_perclass[k]:>12.4f}")

## 10. Final Results & Prediction Analysis

In [ ]:
# Collect images for visualisation using per-class thresholds
all_images_list = []
with torch.no_grad():
    for images, _ in test_loader:
        all_images_list.append(images)
all_images = torch.cat(all_images_list)

correct_idx, partial_idx, incorrect_idx = categorize_predictions(test_labels, preds_perclass)
show_prediction_examples(correct_idx,   all_images, test_labels, preds_perclass, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   all_images, test_labels, preds_perclass, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, all_images, test_labels, preds_perclass, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(test_labels, preds_perclass)
plot_confusion_matrices(test_labels, preds_perclass)

## 11. Summary of Gains

The five improvements stack well:

| Technique | Contribution |
|---|---|
| **ASL** | Focuses training on hard positives; reduces easy-negative dominance. |
| **RandAugment + Erasing** | Stronger regularization; model is forced to use distributed cues. |
| **Stratified split** | Fairer evaluation; rare classes appear in val/test. |
| **Weighted sampling** | Rare-combo images seen more often; improves tail-class recall. |
| **Per-class threshold** | Removes the assumption that all classes share the same operating point. |

**Next:** Notebook 8 provides an in-depth interpretability analysis of this
final best model — GradCAM heatmaps, confusion matrices, and error taxonomy.